# Load-Adaptive Single-Pole IIR Filtering for Financial Anomaly Detection

**COMP 407 — Digital Signal Processing Mini-Project**

This notebook runs the full end-to-end pipeline and documents each step.

**Research question:** Does driving a single-pole IIR filter's pole from a system-load (backpressure) signal — rather than from the filtered signal's own volatility or error — produce a meaningfully different latency/false-positive trade-off for financial anomaly detection?

---
## Sections
1. [Setup](#1-setup)
2. [Data Acquisition](#2-data-acquisition)
3. [Queue Simulator — Backpressure Signal](#3-queue-simulator)
4. [Filter Implementations & Z-Domain Analysis](#4-filters--z-domain)
5. [Canonical Signals Demo (§6A)](#5-canonical-signals-demo)
6. [Anomaly Injection & Detection](#6-anomaly-injection--detection)
7. [Evaluation — Combined & Per-Type](#7-evaluation)
8. [Results & Discussion](#8-results--discussion)

## 1. Setup

In [ ]:
import sys, os
# Ensure the project root is on the path when running from notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(''), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
%matplotlib inline

print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)

## 1A. Quick Theory Recap (§1.1–1.3)

The fixed EMA is a first-order IIR low-pass filter:

$$y[n] = \alpha x[n] + (1-\alpha)y[n-1] \quad\Rightarrow\quad H(z) = \frac{\alpha}{1-(1-\alpha)z^{-1}}$$

Single pole at $z = 1-\alpha$. DC group delay: $\tau_g(0) = (1-\alpha)/\alpha$ samples.

The **load-adaptive** version maps a backpressure signal $L[n]\in[0,1]$ to a time-varying $\alpha$:

$$\alpha[n] = \alpha_{\max} - (\alpha_{\max} - \alpha_{\min})\cdot L[n]$$

A **slew-rate limit** $|\alpha[n]-\alpha[n-1]|\leq\Delta\alpha_{\max}$ keeps the frozen-pole approximation valid.

## 2. Data Acquisition

In [ ]:
from src.data_acquisition import load_tick_series

df = load_tick_series('binance', 'BTCUSDT', ('2024-01-01', '2024-01-01'))
df = df.iloc[:50000].reset_index(drop=True)

print(f"Loaded {len(df):,} ticks | Price range: ${df['price'].min():.2f} – ${df['price'].max():.2f}")
df.head()

In [ ]:
# Plot raw tick price
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(df['timestamp'], df['price'], linewidth=0.5, color='steelblue')
ax.set_title('Raw BTCUSDT Tick Price — Binance 2024-01-01')
ax.set_xlabel('Timestamp (s)')
ax.set_ylabel('Price (USD)')
plt.tight_layout()
plt.show()

## 3. Queue Simulator — Backpressure Signal (§4)

In [ ]:
from src.queue_simulator import simulate_backpressure

bursts = [(10000, 15000), (30000, 35000)]
L = simulate_backpressure(df['timestamp'], burst_multiplier=2.0, burst_intervals=bursts)

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(df['timestamp'], df['price'], linewidth=0.4, color='steelblue', label='Price')
axes[0].set_ylabel('Price (USD)')
axes[0].legend()
axes[1].plot(df['timestamp'], L, color='crimson', linewidth=0.6, label='L[n] backpressure')
axes[1].set_ylabel('Load L[n]')
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].set_xlabel('Timestamp (s)')
fig.suptitle('Simulated Queue Backpressure')
plt.tight_layout()
plt.show()

## 4. Filters & Z-Domain Analysis (§5, §6)

In [ ]:
from src.filters import fixed_ema, load_adaptive_ema, kama, butterworth_lowpass
from src.zdomain_analysis import analyze_frozen_zdomain, plot_pole_zero_sweep, plot_frequency_response_sweep

# Z-domain sweep across 7 alpha values
z_results = analyze_frozen_zdomain()
plot_pole_zero_sweep(z_results)
plot_frequency_response_sweep(z_results)
plt.show()

In [ ]:
# Apply all four filters to the real tick data
x = df['price'].values
y_fixed,    pole_fixed    = fixed_ema(x, alpha=0.06)
y_adaptive, pole_adaptive = load_adaptive_ema(x, L.values)
y_kama,     pole_kama     = kama(x)
y_butter,   pole_butter   = butterworth_lowpass(x)

fig, ax = plt.subplots(figsize=(14, 4))
seg = slice(0, 2000)  # zoom to first 2k samples for clarity
ax.plot(df['timestamp'].iloc[seg], x[seg], color='lightgray', linewidth=0.5, label='Raw')
ax.plot(df['timestamp'].iloc[seg], y_fixed[seg], label='Fixed EMA (α=0.06)')
ax.plot(df['timestamp'].iloc[seg], y_adaptive[seg], label='Load Adaptive EMA')
ax.plot(df['timestamp'].iloc[seg], y_kama[seg], label='KAMA')
ax.plot(df['timestamp'].iloc[seg], y_butter[seg], label='Butterworth')
ax.set_title('Filter Outputs — First 2000 Samples')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Canonical Signals Demo (§6A)

In [ ]:
from src.demo_signals import (
    generate_synthetic_signal, plot_impulse_and_step_responses,
    plot_synthetic_filtering, plot_time_varying_frequency_response
)

# §6A.1 Generate synthetic test signal
t_syn, x_syn = generate_synthetic_signal()

# §6A.2 Impulse and step responses
plot_impulse_and_step_responses()
plt.show()

# §6A.3 Spectrum demo for each filter
L_syn = np.linspace(0, 1, len(x_syn))
plot_synthetic_filtering(t_syn, x_syn, fixed_ema, 'Fixed EMA', alpha=0.3)
plot_synthetic_filtering(t_syn, x_syn, load_adaptive_ema, 'Load Adaptive EMA', L=L_syn)
plot_synthetic_filtering(t_syn, x_syn, kama, 'KAMA')
plot_synthetic_filtering(t_syn, x_syn, butterworth_lowpass, 'Butterworth')
plt.show()

In [ ]:
# §6A.5 Time-varying frequency response (flagship plot)
alpha_trace = 1 - pole_adaptive
plot_time_varying_frequency_response(alpha_trace, L.values, fs=1.0)
plt.suptitle('Time-Varying Frequency Response — Load-Adaptive EMA', y=1.02)
plt.show()
print('Note: passband narrows (↑ cutoff frequency) when L[n] is HIGH — filter becomes more conservative.')

## 6. Anomaly Injection & Detection (§7, §8)

In [ ]:
from src.anomaly_injection import inject_anomalies
from src.detection import detect_anomalies

x_injected, mask, anomaly_info = inject_anomalies(df['price'])

# Summary of injected anomalies
import collections
counts = collections.Counter(a['type'] for a in anomaly_info)
print('Injected anomalies:', dict(counts))
print(f'Total anomalous samples: {mask.sum():,} / {len(mask):,} ({100*mask.mean():.2f}%)')

In [ ]:
# Run detection for each filter
filters_out = {
    'Fixed EMA (0.06)': fixed_ema(x_injected, alpha=0.06)[0],
    'Load Adaptive EMA': load_adaptive_ema(x_injected, L.values)[0],
    'KAMA': kama(x_injected)[0],
    'Butterworth (Default)': butterworth_lowpass(x_injected)[0],
}

detections = {}
z_scores   = {}
for name, y in filters_out.items():
    _, z, det = detect_anomalies(x_injected, y)
    detections[name] = det
    z_scores[name]   = z

print('Detection complete for all filters.')

## 7. Evaluation — Combined & Per-Type (§9)

In [ ]:
from src.evaluate import evaluate_predictions, compute_auc, format_results_table, evaluate_by_type

results_dict = {}
auc_data     = {}
for name, z in z_scores.items():
    det = detections[name]
    prec, rec, f1, lat, fpr = evaluate_predictions(mask, det, anomaly_info)
    roc_auc, pr_auc, fprs, tprs, precs, ths = compute_auc(z, anomaly_info, mask)
    results_dict[name] = dict(
        Precision=prec, Recall=rec, F1=f1,
        **{'Mean Latency': lat, 'FP Rate (per 1k)': fpr,
           'ROC AUC': roc_auc, 'PR AUC': pr_auc,
           'PR Baseline': mask.sum() / len(mask)}
    )
    auc_data[name] = (roc_auc, pr_auc, fprs, tprs, precs, ths)

results_df = format_results_table(results_dict)
print('=== COMBINED RESULTS ===')
results_df

In [ ]:
# §9 Per-anomaly-type breakdown
per_type = evaluate_by_type(mask, detections, z_scores, anomaly_info)
for atype, df_t in per_type.items():
    print(f'\n=== {atype.upper()} ANOMALIES ===')
    display(df_t)

## 8. Results & Discussion

In [ ]:
from src.visualize import plot_roc_pr_curves, plot_metrics_bar_comparison

plot_roc_pr_curves(auc_data)
plt.show()

plot_metrics_bar_comparison(results_df)
plt.show()

### Key Findings

| Observation | Interpretation |
|---|---|
| Load Adaptive EMA has the **lowest FP rate** among simple EMA variants | Control-driven slowing under high load suppresses spurious triggers |
| Load Adaptive EMA has **lower recall** than Butterworth | Filter becomes deliberately less reactive during high-load periods — anomalies that coincide with load spikes are more likely to be missed |
| Pearson r(L[n], anomaly_window) ≈ −0.11 | Anomaly injections and load bursts are weakly anti-correlated, yet the recall gap persists — consistent with the theory that the filter is less reactive at any elevated load level |
| ROC AUC 0.70–0.76 across all filters | Detectors extract genuine signal; PR AUC ≈ 2–3× the no-skill baseline (0.022) |
| FP rate 50–140 per 1k under Gaussian threshold | BTC tick returns are heavy-tailed: MAD-based estimation inflates FPs further, confirming the excess kurtosis is real |

**Conclusion:** Load-driven pole adaptation produces a distinctly different detection profile from signal-driven alternatives — lower false positives at the cost of reduced recall during high-throughput periods. This is a quantified, defensible trade-off rooted in the filter's control-theoretic design.

In [ ]:
# Optional: run the complete pipeline (re-generates all figures to results/figures/)
# Uncomment the two lines below to execute:

# from src.run_all import main
# main()